# 02 — Degradation Recovery & Noise Model  *(Blocks E & F)*

KLA permits creating extra synthetic pairs from the provided GT images (§4B).
If we can reproduce their degradation process, we can generate unlimited training
data — the main defence against unfamiliar test content.

Two complications straight from the spec:

* *"The three degradations may have been applied in any order"* → order is a
  **per-image** property, not a single global answer.
* *"sampled levels may vary within a similar range"* → keep synthetic jitter
  modest. Over-widening teaches the model to hedge, and hedging looks like blur.

**Timebox this notebook.** If the kernel does not fall out in ~3 hours, take the
least-squares fallback at the bottom and move on.

In [ ]:
import sys, os
REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO)
%load_ext autoreload
%autoreload 2

import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from src.io_utils import load_image, pair_by_stem
from src.degrade import downsample
from src.transforms import load_stats, save_stats

DATA = Path("/kaggle/input/kla-dataset")
pairs = pair_by_stem(DATA/"GT", DATA/"NoisyLR")
rng = np.random.default_rng(0)
sample = [pairs[i] for i in rng.choice(len(pairs), min(24, len(pairs)), replace=False)]
print(len(pairs), "pairs |", len(sample), "sampled for fitting")

## 1. Which downsampling kernel?

The observed LR is noisy, so comparing directly is dominated by noise. Compare in
a **low-frequency band** instead, where the kernel differences live and the noise
does not.

In [ ]:
from scipy.ndimage import gaussian_filter

def lowpass(x, s=2.0):
    return gaussian_filter(x.astype(np.float32), s)

KERNELS = ["bicubic_aa", "bilinear_aa", "area", "lanczos", "nearest"]

scores = {}
for k in KERNELS:
    errs = []
    for gt_p, lr_p in sample:
        gt, lr = load_image(gt_p), load_image(lr_p)
        try:
            est = downsample(gt, k)
        except Exception as e:
            errs = [np.inf]; break
        if est.shape != lr.shape:      # guard against odd sizes
            continue
        errs.append(float(np.mean((lowpass(est) - lowpass(lr))**2)))
    scores[k] = float(np.mean(errs)) if errs else np.inf

for k, v in sorted(scores.items(), key=lambda kv: kv[1]):
    print(f"{k:<14} lowpass MSE = {v:.6e}")
best_kernel = min(scores, key=scores.get)
print("\nbest:", best_kernel)

### Sub-pixel alignment

A half-pixel offset (the classic `align_corners` mismatch) silently caps SSIM for
the entire project. Check it once.

In [ ]:
from skimage.registration import phase_cross_correlation

shifts = []
for gt_p, lr_p in sample[:12]:
    gt, lr = load_image(gt_p), load_image(lr_p)
    est = downsample(gt, best_kernel)
    if est.shape != lr.shape: continue
    s, _, _ = phase_cross_correlation(est, lowpass(lr), upsample_factor=20)
    shifts.append(s)
shifts = np.array(shifts)
print("median sub-pixel shift (dy, dx):", np.median(shifts, axis=0))
print("-> anything near +/-0.5 means an alignment convention mismatch to fix")

## 2. Noise model

With a recovered kernel we get a clean LR reference, so the noise residual is exact:

$$r = \text{LR}_{\text{noisy}} - \text{downsample}(\text{GT})$$

Fit $\mathrm{var}(r) = a\mu^2 + b\mu + c$:

* **a** dominant → multiplicative (speckle) → a log transform makes it additive
* **b** dominant → Poisson / shot
* **c** dominant → additive Gaussian

The spec says speckle *and* Gaussian are both present, so expect a mixture.

In [ ]:
from skimage.measure import block_reduce
from scipy.optimize import nnls

mus, varis, residuals, cleans = [], [], [], []
for gt_p, lr_p in sample:
    gt, lr = load_image(gt_p), load_image(lr_p)
    clean = downsample(gt, best_kernel)
    if clean.shape != lr.shape: continue
    r = lr - clean
    residuals.append(r); cleans.append(clean)
    mus.append(block_reduce(clean, 8, np.mean).ravel())
    varis.append(block_reduce(r, 8, np.var).ravel())

mu  = np.concatenate(mus)
var = np.concatenate(varis)
m   = np.isfinite(mu) & np.isfinite(var)

# Non-negative least squares: variances cannot be negative, and an unconstrained
# polyfit happily returns a negative additive term that then breaks sqrt().
A = np.stack([mu[m]**2, mu[m], np.ones(m.sum())], axis=1)
(a, b, cst), _ = nnls(A, var[m])

print(f"var(r) = {a:.5e}*mu^2 + {b:.5e}*mu + {cst:.5e}")
print(f"  multiplicative (a): {a:.3e}   -> sigma_mult ~ {np.sqrt(a):.4f}")
print(f"  poisson        (b): {b:.3e}")
print(f"  additive       (c): {cst:.3e}   -> sigma_add  ~ {np.sqrt(cst):.4f}")
print("\nNote: the three terms are correlated, so individual coefficients are")
print("only indicative. What matters is that degrade() reproduces the observed")
print("residual statistics - verified in section 3 below.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

idx = np.random.default_rng(0).choice(mu[m].size, min(20000, mu[m].size), replace=False)
ax[0].scatter(mu[m][idx], var[m][idx], s=2, alpha=.15)
xs = np.linspace(mu[m].min(), mu[m].max(), 100)
ax[0].plot(xs, a*xs**2 + b*xs + cst, "r-", lw=2)
ax[0].set_xlabel("local mean"); ax[0].set_ylabel("local variance")
ax[0].set_title("variance vs signal (slope => multiplicative)")

R = np.concatenate([r.ravel() for r in residuals])
ax[1].hist(R, bins=200, density=True)
ax[1].set_title(f"residual histogram (std={R.std():.4f})"); ax[1].set_yscale("log")

ac = [np.corrcoef(r[:, :-1].ravel(), r[:, 1:].ravel())[0,1] for r in residuals]
ax[2].hist(ac, bins=30)
ax[2].set_title("noise spatial autocorrelation, per image")
ax[2].set_xlabel("lag-1 correlation")
plt.tight_layout(); plt.show()

### Degradation order — per image

The obvious test is the spatial autocorrelation of the residual: noise added
*before* downsampling gets averaged and becomes correlated, noise added *after*
stays white.

**That test is weak at 2x** — the resampling kernel's support is small relative to
the decimation, so lag-1 correlation stays near zero either way. Verified on
synthetic data where the true mix was known: the autocorrelation test reported 0%
noise-first when the truth was 40%.

So the primary method here is **forward-simulation hypothesis testing**: simulate
both orders with the fitted noise parameters, and pick whichever reproduces a
vector of observed residual statistics more closely. Autocorrelation is kept as a
secondary signal only.

In [ ]:
from src.degrade import add_noise

sigma_mult, sigma_add = float(np.sqrt(a)), float(np.sqrt(cst))

def summary(r, clean):
    """Statistics that separate 'noise before downsample' from 'after'."""
    g  = np.hypot(*np.gradient(clean))
    hi = g > np.quantile(g, 0.75)
    return np.array([
        r.std(),                                                    # magnitude
        r[hi].std() / (r[~hi].std() + 1e-9),                        # texture dependence
        np.corrcoef(r[:, :-1].ravel(), r[:, 1:].ravel())[0, 1],     # spatial correlation
        np.corrcoef(np.abs(r).ravel(), clean.ravel())[0, 1],        # multiplicative signature
    ])

def simulate(gt, order, k, sm, sa, rng):
    if order == "noise_first":
        return downsample(add_noise(gt, sm, sa, rng), k)
    return add_noise(downsample(gt, k), sm, sa, rng)

votes, margins = [], []
for gt_p, lr_p in sample:
    gt, lr = load_image(gt_p), load_image(lr_p)
    clean = downsample(gt, best_kernel)
    if clean.shape != lr.shape: continue
    obs = summary(lr - clean, clean)
    dist = {}
    for order in ("noise_first", "noise_last"):
        sims = [summary(simulate(gt, order, best_kernel, sigma_mult, sigma_add,
                                 np.random.default_rng(s)) - clean, clean)
                for s in range(5)]
        dist[order] = float(np.linalg.norm((np.mean(sims, 0) - obs) / (np.abs(obs) + 1e-6)))
    votes.append(min(dist, key=dist.get))
    margins.append(abs(dist["noise_first"] - dist["noise_last"]))

frac_first = votes.count("noise_first") / max(len(votes), 1)
order_mix = [round(frac_first, 3), round(1 - frac_first, 3)]   # [noise_first, noise_last]

print(f"forward-simulation vote: {frac_first:.0%} noise_first / {1-frac_first:.0%} noise_last")
print(f"median decision margin : {np.median(margins):.3f}  (near 0 => the two are indistinguishable)")

ac = np.array([np.corrcoef(r[:, :-1].ravel(), r[:, 1:].ravel())[0, 1] for r in residuals])
print(f"secondary (autocorr)   : median {np.median(ac):+.4f}, "
      f"{np.mean(ac > 0.2):.0%} above 0.2")
print("\norder_mix =", order_mix)
print("If the margin is tiny, the order genuinely does not matter much -")
print("just randomise it 50/50 in degrade.py and move on.")

### Log-transform decision

Only worth it if the multiplicative term genuinely dominates. It costs a little
inference time, and inference time is scored.

In [ ]:
mu_typ = float(np.median(mu[m]))
contrib = {"multiplicative": a*mu_typ**2, "poisson": b*mu_typ, "additive": cst}
tot = sum(abs(v) for v in contrib.values())
for k, v in contrib.items():
    print(f"{k:<16} {v:.3e}   ({100*abs(v)/tot:5.1f}% of variance at typical intensity)")

use_log = abs(contrib["multiplicative"]) > 0.5 * tot
print("\n-> log_transform =", use_log)

In [ ]:
# Persist findings. src/degrade.py and the training config read these.
stats = load_stats("../artifacts/stats.json")
stats.update({
    "log_transform": bool(use_log),
    "downsample_kernel": best_kernel,
    "kernel_scores": {k: float(v) for k, v in scores.items()},
    "noise_var_fit": {"a_mult": float(a), "b_poisson": float(b), "c_additive": float(cst)},
    "order_mix": order_mix,
    "meas_mult": float(np.sqrt(max(a, 0))),
    "meas_add": float(np.sqrt(max(cst, 0))),
    "subpixel_shift": np.median(shifts, axis=0).tolist() if len(shifts) else None,
})
save_stats(stats, "../artifacts/stats.json")
print(json.dumps({k: stats[k] for k in
      ["downsample_kernel","order_mix","meas_mult","meas_add","log_transform"]}, indent=2))
print("\nIf log_transform flipped to True, re-run: python -m pytest tests/ -q")

## 3. Verify the recipe

With `width=0` (no jitter), `degrade()` should reproduce the real NoisyLR
statistics. If it does not, the recipe is wrong — fix it before generating
synthetic training data on top of a bad model.

In [ ]:
from src.degrade import degrade

cfg = dict(width=0.0, jitter=0.30,
           kernels=[best_kernel], kernel_p=[1.0],
           order_mix=order_mix,
           meas_mult=stats["meas_mult"], meas_add=stats["meas_add"])

real_std, synth_std, real_over, synth_over = [], [], [], []
for gt_p, lr_p in sample[:12]:
    gt, lr = load_image(gt_p), load_image(lr_p)
    syn = degrade(gt, np.random.default_rng(1), cfg)
    if syn.shape != lr.shape: continue
    real_std.append((lr - downsample(gt, best_kernel)).std())
    synth_std.append((syn - downsample(gt, best_kernel)).std())
    real_over.append((lr > 1).mean()); synth_over.append((syn > 1).mean())

print(f"residual std   real {np.mean(real_std):.5f}  vs  synthetic {np.mean(synth_std):.5f}")
print(f"frac > 1       real {np.mean(real_over):.5f}  vs  synthetic {np.mean(synth_over):.5f}")
print("\nClose on both lines => the recipe is good enough to generate training data.")

### Fallback if no named kernel wins

Solve for a numeric kernel by least squares. A numeric kernel works perfectly
well in `degrade.py` — you never need to name it.

In [ ]:
def fit_kernel(pairs, ksize=5, scale=2, n=6):
    A, y = [], []
    for gt_p, lr_p in pairs[:n]:
        gt, lr = load_image(gt_p), load_image(lr_p)
        pad = ksize // 2
        gp = np.pad(gt, pad, mode="reflect")
        H, W = lr.shape
        ys, xs = np.random.default_rng(0).integers(0, H, 300), np.random.default_rng(1).integers(0, W, 300)
        for yy, xx in zip(ys, xs):
            r, c0 = yy*scale, xx*scale
            patch = gp[r:r+ksize, c0:c0+ksize]
            if patch.shape != (ksize, ksize): continue
            A.append(patch.ravel()); y.append(lr[yy, xx])
    A, y = np.asarray(A), np.asarray(y)
    k, *_ = np.linalg.lstsq(A, y, rcond=None)
    return k.reshape(ksize, ksize)

# K = fit_kernel(sample); print(np.round(K, 4)); print("sum =", K.sum())